In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product

import time
import sys
import requests
import logging
import os

from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from scipy import stats
from scipy.optimize import minimize
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error,mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from statsmodels.tsa.api import SimpleExpSmoothing, Holt, ExponentialSmoothing

In [2]:
import pandas as pd
import numpy as np

df = pd.read_excel("Agc 23 Dec 25.xlsx", sheet_name="No BTM", skiprows=4)
df["P/N"] = df["P/N"].astype(str)

# -----------------------------
# CONFIG
# -----------------------------
pn_2867 = "CC 2867"
pn_2868 = "CC 2868"
pn_2869 = "CC 2869"

d_cols = [f"D-{i}" for i in range(1, 17)]
call_cols = [c for c in df.columns if c.startswith("C-") or c == "C TM"]

# -----------------------------
# HELPER
# -----------------------------
def get_row(brc, pn):
    rows = df[(df["Brc"] == brc) & (df["P/N"] == pn)]
    if not rows.empty:
        return rows.iloc[0]
    return pd.Series(0, index=df.columns)

# -----------------------------
# BUILD CC 2867A
# -----------------------------
new_rows = []

for brc in df["Brc"].dropna().unique():

    r2867 = get_row(brc, pn_2867)
    r2868 = get_row(brc, pn_2868)
    r2869 = get_row(brc, pn_2869)

    # Start with all blank
    new_row = pd.Series(index=df.columns, dtype="object")

    new_row["Brc"] = brc
    new_row["P/N"] = "CC 2867A"
    new_row["Desc"] = "Total of CC2867"
    new_row["D TM"] = 0

    # DN Price
    new_row["DN Price"] = 3038.49

    # OH
    new_row["OH"] = (
        ((r2869["OH"] * 20) + (r2868["OH"] * 200)) / 1000
    ) + r2867["OH"]

    # OO
    new_row["OO"] = (
        ((r2869["OO"] * 20) + (r2868["OO"] * 200)) / 1000
    ) + r2867["OO"]

    # D-1 to D-16
    for col in d_cols:
        new_row[col] = (
            ((r2869[col] * 20) + (r2868[col] * 200)) / 1000
        ) + r2867[col]

    #Agc
    new_row["Agc"] = 23
    # Book / Alloc
    new_row["Book"] = 0
    new_row["Alloc In"] = 0
    new_row["Alloc Out"] = 0

    # Call columns
    for col in call_cols:
        new_row[col] = (
            ((r2869[col] * 20) + (r2868[col] * 200)) / 1000
        ) + r2867[col]

    new_rows.append(new_row)

# -----------------------------
# APPEND
# -----------------------------
df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)


In [3]:
print(df.columns)

Index(['Brc', 'Agc', 'P/N', 'Desc', 'DN Price', 'OH', 'OO', 'Book', 'Alloc In',
       'Alloc Out', 'D TM', 'D-1', 'D-2', 'D-3', 'D-4', 'D-5', 'D-6', 'D-7',
       'D-8', 'D-9', 'D-10', 'D-11', 'D-12', 'D-13', 'D-14', 'D-15', 'D-16',
       'C TM', 'C-1', 'C-2', 'C-3', 'C-4', 'C-5', 'C-6', 'C-7', 'C-8', 'C-9',
       'C-10', 'C-11', 'C-12', 'Last Sales', 'Last GRR', 'Last Return', 'Date',
       'Category', 'Bin Loc', 'Status', 'brc.pn'],
      dtype='object')


In [4]:
print(df)

       Brc  Agc         P/N                       Desc  DN Price    OH    OO  \
0       20   23  142784   S                       HEAD     24.63  0.00  0.00   
1       20   23      15/25B  HYDRAULIC FLUID SEPARATOR   5190.00  0.00  0.00   
2       20   23  153518   S                  BOLT SEAL      1.21  0.00  0.00   
3       20   23  153520   S                     WASHER     12.64  0.00  0.00   
4       20   23  154272   S                    ELEMENT     18.88  0.00  0.00   
...    ...  ...         ...                        ...       ...   ...   ...   
31147   95   23    CC 2867A            Total of CC2867   3038.49  0.26  0.96   
31148   96   23    CC 2867A            Total of CC2867   3038.49  0.22  0.00   
31149   97   23    CC 2867A            Total of CC2867   3038.49  4.00  0.00   
31150   98   23    CC 2867A            Total of CC2867   3038.49  0.00  0.00   
31151   99   23    CC 2867A            Total of CC2867   3038.49  0.00  0.00   

       Book  Alloc In  Alloc Out  ...  

In [5]:
# =========================================================
# EXPORT
# =========================================================
df.to_excel("Agc 23 Dec 25 Updated.xlsx", index=False)